In [1]:
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

############################################################
# 1️⃣ FnGuide 파싱 함수
############################################################


def load_fnguide_html(gicode):
    url = f'https://comp.fnguide.com/SVO2/ASP/SVD_main.asp?pGB=1&gicode={gicode}'
    html = BeautifulSoup(requests.get(url).text, 'html.parser')
    return html



def parse_yearly_financials(html, target_items):
    tables = html.find_all('table', class_='us_table_ty1 h_fix zigbg_no')
    table_y = tables[5]

    rows = table_y.find_all('tr')
    raw_data = {}

    for row in rows:
        th = row.find('th')
        if not th:
            continue
        item = th.get_text(strip=True)
        if item in target_items:
            raw_data[item] = [
                int(td.get_text(strip=True).replace(',', '')) if td.get_text(strip=True) else None
                for td in row.find_all('td')
            ]

    this_year = int(
        html.find('tr', class_='td_gapcolor2')
            .find('span', class_='txt_acd')
            .get_text()
            .split('/')[0]
    )

    years = list(range(this_year - 5, this_year - 5 + len(raw_data['매출액'])))

    operating_profit_y = [
        raw_data['영업이익(발표기준)'][i] if y < this_year else raw_data['영업이익'][i]
        for i, y in enumerate(years)
    ]

    df_year = pd.DataFrame({
        '연도': years,
        '매출액': raw_data['매출액'],
        '영업이익': operating_profit_y,
        '당기순이익': raw_data['당기순이익']
    })

    return df_year, this_year


############################################################
# 2️⃣ 코스피 + 코스닥 대표 상위 30개 종목코드
############################################################

def get_top30_gicodes():

    tickers = [
        "005930",  # 삼성전자
        "000660",  # SK하이닉스
        "207940",  # 삼성바이오로직스
        "068270",  # 셀트리온
        "005380",  # 현대차
        "000270",  # 기아
        "035420",  # NAVER
        "035720",  # 카카오
        "051910",  # LG화학
        "373220",  # LG에너지솔루션
        "006400",  # 삼성SDI
        "247540",  # 에코프로비엠
        # "086520",  # 에코프로
        "005490",  # 포스코홀딩스
        "066570",  # LG전자
        "015760",  # 한국전력
        "034020",  # 두산에너빌리티
        "042660",  # 한화오션
        "010140",  # 삼성중공업
        "064400",  # LG CNS
        "257720",  # 실리콘투
        "012450",  # 한화에어로스페이스
        "329180",  # 현대중공업
        "009540",  # 한국조선해양
        "009150",  # 삼성전기
        "011070",  # LG이노텍
        "042700",  # 한미반도체
        "058470",  # 리노공업
        "357780",  # 솔브레인
        "090430",  # 아모레퍼시픽
        # "253450",  # 스튜디오드래곤
        # "089860",  # 롯데렌탈
    ]

    return ["A" + code for code in tickers]


############################################################
# 3️⃣ 배치 실행 (1번 옵션 자동)
############################################################

def run_batch_check():

    gicodes = get_top30_gicodes()
    incomplete_stocks = []
    all_results = []

    for gicode in gicodes:

        try:
            html = load_fnguide_html(gicode)
            stock_name = html.find('h1').get_text(strip=True)

            df_year_latest, this_year = parse_yearly_financials(
                html,
                ['매출액', '영업이익', '영업이익(발표기준)', '당기순이익']
            )

            # 1️⃣ FnGuide 최신 실적 자동 사용
            df_final = df_year_latest.copy()

            if df_final[['매출액','영업이익','당기순이익']].isna().any().any():
                incomplete_stocks.append(stock_name)
            else:
                df_final['종목코드'] = gicode
                df_final['종목명'] = stock_name
                all_results.append(df_final)

            print(f"✅ 완료: {stock_name}")

        except Exception as e:
            print(f"❌ 오류 발생: {gicode} → {e}")
            incomplete_stocks.append(gicode)

    if all_results:
        final_df = pd.concat(all_results, ignore_index=True)
    else:
        final_df = pd.DataFrame()

    return final_df, incomplete_stocks


############################################################
# 4️⃣ 실행
############################################################

final_df, incomplete_list = run_batch_check()

print("\n" + "="*60)
print("🚨 전망치 미완성 종목")
print("="*60)

for name in incomplete_list:
    print(name)

print("\n" + "="*60)
print("✅ 정상 종목 데이터 샘플")
print("="*60)

final_df.head()


✅ 완료: 삼성전자
✅ 완료: SK하이닉스
✅ 완료: 삼성바이오로직스
✅ 완료: 셀트리온
✅ 완료: 현대차
✅ 완료: 기아
✅ 완료: NAVER
✅ 완료: 카카오
✅ 완료: LG화학
✅ 완료: LG에너지솔루션
✅ 완료: 삼성SDI
✅ 완료: 에코프로비엠
✅ 완료: POSCO홀딩스
✅ 완료: LG전자
✅ 완료: 한국전력
✅ 완료: 두산에너빌리티
✅ 완료: 한화오션
✅ 완료: 삼성중공업
✅ 완료: LG씨엔에스
✅ 완료: 실리콘투
✅ 완료: 한화에어로스페이스
✅ 완료: HD현대중공업
✅ 완료: HD한국조선해양
✅ 완료: 삼성전기
✅ 완료: LG이노텍
✅ 완료: 한미반도체
✅ 완료: 리노공업
✅ 완료: 솔브레인
✅ 완료: 아모레퍼시픽

🚨 전망치 미완성 종목
삼성바이오로직스
NAVER
LG화학
리노공업

✅ 정상 종목 데이터 샘플


,연도,매출액,영업이익,당기순이익,종목코드,종목명
0,2021,2796048,516339,399075,A005930,삼성전자
1,2022,3022314,433766,556541,A005930,삼성전자
2,2023,2589355,65670,154871,A005930,삼성전자
3,2024,3008709,327260,344514,A005930,삼성전자
4,2025,3336059,436011,452068,A005930,삼성전자


In [2]:
import numpy as np
import pandas as pd

############################################################
# 성장률 함수 (기존 그대로)
############################################################

def calc_3y_avg_growth(series):

    s = series.dropna()

    if len(s) < 4:
        return np.nan

    start = s.iloc[0]
    end = s.iloc[-1]

    if end < 0:
        return np.nan

    if start > 0 and end > 0:
        return (end / start) ** (1 / 3) - 1

    min_val = s.min()
    scale = np.mean(np.abs(s))

    if scale == 0:
        return np.nan

    shift = abs(min_val) + scale

    shifted_start = start + shift
    shifted_end = end + shift

    if shifted_start <= 0:
        return np.nan

    return (shifted_end / shifted_start) ** (1 / 3) - 1


############################################################
# 시가총액 / PER
############################################################

def get_market_cap(gicode):
    url = f'https://comp.fnguide.com/SVO2/ASP/SVD_main.asp?pGB=1&gicode={gicode}'
    headers = {"User-Agent": "Mozilla/5.0"}

    html = BeautifulSoup(requests.get(url, headers=headers).text, 'html.parser')

    try:
        cap_text = (
            html.find('th', string='시가총액')
                .find_next_sibling('td')
                .get_text(strip=True)
        )
        return float(cap_text.replace(',', ''))
    except:
        return np.nan


def calc_per(market_cap, profit):
    if pd.isna(market_cap) or pd.isna(profit):
        return np.nan
    if profit <= 0 or market_cap <= 0:
        return np.nan

    return market_cap / profit


############################################################
# 🔥 final_df 기반 전체 종목 계산
############################################################

result_rows = []

for gicode in final_df['종목코드'].unique():

    df_stock = (
        final_df[final_df['종목코드'] == gicode]
        .sort_values('연도')
    )

    stock_name = df_stock['종목명'].iloc[0]

    # 최근 4개 연도
    recent_4y = df_stock.tail(4)

    if len(recent_4y) < 4:
        continue

    growth_sales = calc_3y_avg_growth(recent_4y['매출액'])
    growth_op = calc_3y_avg_growth(recent_4y['영업이익'])
    growth_net = calc_3y_avg_growth(recent_4y['당기순이익'])


    future_row = recent_4y.iloc[-1]

    market_cap = get_market_cap(gicode)

    per_operating = calc_per(market_cap, future_row['영업이익'])
    per_net = calc_per(market_cap, future_row['당기순이익'])

    result_rows.append({
        "종목코드": gicode,
        "종목명": stock_name,
        "작년 영업이익": recent_4y['영업이익'].iloc[0],
        "작년 당기순이익": recent_4y['당기순이익'].iloc[0],
        "내후년 영업이익(E)": future_row['영업이익'],
        "내후년 당기순이익(E)": future_row['당기순이익'],
        "매출액_3Y성장률": growth_sales,
        "영업이익_3Y성장률": growth_op,
        "순이익_3Y성장률": growth_net,
        "시가총액": market_cap,
        "영업이익_PER": per_operating,
        "순이익_PER": per_net
    })

    print(f"✅ 완료: {stock_name}")

result_df = pd.DataFrame(result_rows)

pd.set_option('display.float_format', '{:.2f}'.format)

print("\n📊 종목별 성장률 + PER 결과")
result_df


✅ 완료: 삼성전자
✅ 완료: SK하이닉스
✅ 완료: 셀트리온
✅ 완료: 현대차
✅ 완료: 기아
✅ 완료: 카카오
✅ 완료: LG에너지솔루션
✅ 완료: 삼성SDI
✅ 완료: 에코프로비엠
✅ 완료: POSCO홀딩스
✅ 완료: LG전자
✅ 완료: 한국전력
✅ 완료: 두산에너빌리티
✅ 완료: 한화오션
✅ 완료: 삼성중공업
✅ 완료: LG씨엔에스
✅ 완료: 실리콘투
✅ 완료: 한화에어로스페이스
✅ 완료: HD현대중공업
✅ 완료: HD한국조선해양
✅ 완료: 삼성전기
✅ 완료: LG이노텍
✅ 완료: 한미반도체
✅ 완료: 솔브레인
✅ 완료: 아모레퍼시픽

📊 종목별 성장률 + PER 결과


,종목코드,종목명,작년 영업이익,작년 당기순이익,내후년 영업이익(E),내후년 당기순이익(E),매출액_3Y성장률,영업이익_3Y성장률,순이익_3Y성장률,시가총액,영업이익_PER,순이익_PER
0,A005930,삼성전자,436011,452068,2199674,1844235,0.24,0.72,0.60,11247312.00,5.11,6.10
1,A000660,SK하이닉스,472063,429479,1854380,1578300,0.58,0.58,0.54,6806308.00,3.67,4.31
2,A068270,셀트리온,4920,4189,20078,16520,0.19,0.60,0.58,482891.00,24.05,29.23
3,A005380,현대차,114679,103648,150780,134330,0.05,0.10,0.09,1085216.00,7.20,8.08
4,A000270,기아,126671,97750,111721,91698,0.06,-0.04,-0.02,632469.00,5.66,6.90
5,A035720,카카오,4602,-1619,11106,8746,0.07,0.34,0.41,225368.00,20.29,25.77
6,A373220,LG에너지솔루션,5754,3386,40056,28125,0.10,0.91,1.03,864630.00,21.59,30.74
7,A006400,삼성SDI,-17224,-5849,19493,19810,0.21,0.56,0.54,322342.00,16.54,16.27
8,A247540,에코프로비엠,-341,-585,2006,1173,0.14,0.45,0.52,193206.00,96.31,164.71
9,A005490,POSCO홀딩스,21736,9476,39123,23565,0.01,0.22,0.35,279219.00,7.14,11.85


In [3]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

############################################################
# 1️⃣ 데이터 준비
############################################################

df = result_df.copy()

# 퍼센트 변환
df['영업이익_3Y성장률_%'] = df['영업이익_3Y성장률'] * 100
df['순이익_3Y성장률_%'] = df['순이익_3Y성장률'] * 100

# 상위 10개 추출
top_op_growth = df.sort_values('영업이익_3Y성장률_%', ascending=False).head(10)
top_net_growth = df.sort_values('순이익_3Y성장률_%', ascending=False).head(10)

# PER는 낮은 순이 저평가
top_op_per = df.sort_values('영업이익_PER', ascending=True).head(10)
top_net_per = df.sort_values('순이익_PER', ascending=True).head(10)

############################################################
# 3️⃣ 2x2 대시보드 생성
############################################################

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "영업이익 3Y 성장률 TOP10",
        "당기순이익 3Y 성장률 TOP10",
        "영업이익 PER (저평가 TOP10)",
        "당기순이익 PER (저평가 TOP10)"
    )
)

############################
# (1) 영업이익 성장률
############################

fig.add_trace(
    go.Bar(
        x=top_op_growth['종목명'],
        y=top_op_growth['영업이익_3Y성장률_%'],
        text=top_op_growth['영업이익_3Y성장률_%'].round(1).astype(str) + '%',
        textposition='outside',
        marker_color='steelblue'
    ),
    row=1, col=1
)

############################
# (2) 순이익 성장률
############################

fig.add_trace(
    go.Bar(
        x=top_net_growth['종목명'],
        y=top_net_growth['순이익_3Y성장률_%'],
        text=top_net_growth['순이익_3Y성장률_%'].round(1).astype(str) + '%',
        textposition='outside',
        marker_color='steelblue'
    ),
    row=1, col=2
)

############################
# (3) 영업이익 PER
############################

fig.add_trace(
    go.Bar(
        x=top_op_per['종목명'],
        y=top_op_per['영업이익_PER'],
        text=top_op_per['영업이익_PER'].round(1),
        textposition='outside',
        marker_color='steelblue'
    ),
    row=2, col=1
)

############################
# (4) 순이익 PER
############################

fig.add_trace(
    go.Bar(
        x=top_net_per['종목명'],
        y=top_net_per['순이익_PER'],
        text=top_net_per['순이익_PER'].round(1),
        textposition='outside',
        marker_color='steelblue'
    ),
    row=2, col=2
)

############################################################
# 4️⃣ 레이아웃 설정
############################################################

fig.update_layout(
    height=900,
    width=1000,
    title_text="📊 성장률 & PER 상위 10개 대시보드",
    showlegend=False
)

fig.update_xaxes(tickangle=-45)

fig.show()


In [4]:
import pandas as pd
import numpy as np

df = result_df.copy()

############################################################
# 1️⃣ Percentile 점수 계산 (0~1)
############################################################

# 성장률 (높을수록 좋음)
df['P_영업성장'] = df['영업이익_3Y성장률'].rank(pct=True)
df['P_순이익성장'] = df['순이익_3Y성장률'].rank(pct=True)

# PER (낮을수록 좋음 → 역순위)
df['P_영업PER'] = 1 - df['영업이익_PER'].rank(pct=True)
df['P_순이익PER'] = 1 - df['순이익_PER'].rank(pct=True)

############################################################
# 2️⃣ 가중치 적용 (성장 중심 전략)
############################################################

# 기여도 계산
df['기여_영업성장'] = 0.2 * df['P_영업성장']
df['기여_순이익성장'] = 0.2 * df['P_순이익성장']
df['기여_영업PER'] = 0.3 * df['P_영업PER']
df['기여_순이익PER'] = 0.3 * df['P_순이익PER']

df['종합점수'] = (
    df['기여_영업성장'] +
    df['기여_순이익성장'] +
    df['기여_영업PER'] +
    df['기여_순이익PER']
)

# 100점 만점 변환
df['종합점수_100'] = df['종합점수'] * 100

############################################################
# 3️⃣ 랭킹
############################################################

df = df.sort_values('종합점수_100', ascending=False)
df['랭킹'] = range(1, len(df)+1)

df['성장점수'] = (df['P_영업성장'] + df['P_순이익성장']) / 2
df['저평가점수'] = (df['P_영업PER'] + df['P_순이익PER']) / 2

def classify(row):
    if row['성장점수'] > 0.6 and row['저평가점수'] > 0.6:
        return "🔥 고성장 + 저평가"
    elif row['성장점수'] > row['저평가점수']:
        return "📈 성장형"
    else:
        return "💎 저평가형"

df['투자스타일'] = df.apply(classify, axis=1)

df[['종목명','종합점수_100','투자스타일']].head(10)

,종목명,종합점수_100,투자스타일
1,SK하이닉스,84.00,🔥 고성장 + 저평가
0,삼성전자,83.60,🔥 고성장 + 저평가
19,HD한국조선해양,83.20,🔥 고성장 + 저평가
11,한국전력,77.60,💎 저평가형
10,LG전자,67.20,💎 저평가형
14,삼성중공업,66.40,📈 성장형
16,실리콘투,59.20,💎 저평가형
18,HD현대중공업,56.00,📈 성장형
9,POSCO홀딩스,52.40,💎 저평가형
7,삼성SDI,51.60,📈 성장형


In [5]:
import plotly.express as px

bubble_fig = px.scatter(
    df,
    x='저평가점수',
    y='성장점수',
    size='종합점수_100',
    color='종합점수_100',
    hover_name='종목명',
    title='📊 Percentile 기반 성장 vs 저평가 버블차트',
    size_max=50
)

bubble_fig.update_layout(
    xaxis_title='저평가 점수 (오른쪽이 저평가)',
    yaxis_title='성장 점수 (위가 고성장)',
    height=700,
    width=1000
)

bubble_fig.show()


In [6]:
import plotly.graph_objects as go
import numpy as np

############################################################
# 1️⃣ 데이터 준비
############################################################

top10 = df.sort_values('종합점수', ascending=False).head(10)

heatmap_df = top10[['종목명','P_영업성장','P_순이익성장','P_영업PER','P_순이익PER']]
heatmap_df = heatmap_df.set_index('종목명')

z = heatmap_df.values
text = np.round(z, 2)

############################################################
# 2️⃣ 텍스트 색 기준
############################################################

white_mask = z > 0.6
black_mask = ~white_mask

############################################################
# 3️⃣ 히트맵 생성
############################################################

heatmap_fig = go.Figure()

# 🎨 컬러용 히트맵
heatmap_fig.add_trace(go.Heatmap(
    z=z,
    x=heatmap_df.columns,
    y=heatmap_df.index,
    colorscale=[
        [0, "#f7fbff"],
        [0.5, "#6baed6"],
        [1, "#08306b"]
    ],
    showscale=True,
    hovertemplate="종목: %{y}<br>팩터: %{x}<br>점수: %{z:.2f}<extra></extra>"
))

# ⚪ 흰색 텍스트 레이어
heatmap_fig.add_trace(go.Scatter(
    x=np.tile(heatmap_df.columns, len(heatmap_df.index))[white_mask.flatten()],
    y=np.repeat(heatmap_df.index, len(heatmap_df.columns))[white_mask.flatten()],
    mode="text",
    text=text.flatten()[white_mask.flatten()],
    textfont=dict(color="white", size=13),
    showlegend=False
))

# ⚫ 검정 텍스트 레이어
heatmap_fig.add_trace(go.Scatter(
    x=np.tile(heatmap_df.columns, len(heatmap_df.index))[black_mask.flatten()],
    y=np.repeat(heatmap_df.index, len(heatmap_df.columns))[black_mask.flatten()],
    mode="text",
    text=text.flatten()[black_mask.flatten()],
    textfont=dict(color="black", size=13),
    showlegend=False
))

heatmap_fig.update_layout(
    title="📊 TOP10 멀티팩터 점수 히트맵 (Auto Text Contrast)",
    height=700,
    width=1000,
    yaxis=dict(autorange="reversed")
)

heatmap_fig.show()


In [7]:
import pandas as pd

############################################################
# 1️⃣ 투자 대상 선정
############################################################

TOP_N = 10
INVEST_AMOUNT = 10_000_000  # 1000만원

portfolio_df = df.sort_values('종합점수', ascending=False).head(TOP_N).copy()

############################################################
# 2️⃣ 종합점수 비례 가중치 계산
############################################################

total_score = portfolio_df['종합점수'].sum()

portfolio_df['비중'] = portfolio_df['종합점수'] / total_score
portfolio_df['투자금액'] = portfolio_df['비중'] * INVEST_AMOUNT

############################################################
# 3️⃣ 보기 좋게 정리
############################################################

portfolio_df['비중(%)'] = (portfolio_df['비중'] * 100).round(2)
portfolio_df['투자금액'] = portfolio_df['투자금액'].round(0)

portfolio_df = portfolio_df[
    ['종목명','종합점수','비중(%)','투자금액','투자스타일']
]

portfolio_df


,종목명,종합점수,비중(%),투자금액,투자스타일
1,SK하이닉스,0.84,12.33,1233118.00,🔥 고성장 + 저평가
0,삼성전자,0.84,12.27,1227246.00,🔥 고성장 + 저평가
19,HD한국조선해양,0.83,12.21,1221374.00,🔥 고성장 + 저평가
11,한국전력,0.78,11.39,1139166.00,💎 저평가형
10,LG전자,0.67,9.86,986494.00,💎 저평가형
14,삼성중공업,0.66,9.75,974750.00,📈 성장형
16,실리콘투,0.59,8.69,869055.00,💎 저평가형
18,HD현대중공업,0.56,8.22,822079.00,📈 성장형
9,POSCO홀딩스,0.52,7.69,769231.00,💎 저평가형
7,삼성SDI,0.52,7.57,757487.00,📈 성장형


In [8]:
# %pip install dash
# %pip install dash_bootstrap_components
# %pip install dash_html_components
# %pip install dash_core_components

In [9]:
# import dash
# from dash import html
# import dash_bootstrap_components as dbc
# import dash_core_components as dcc
# from google.colab import output

# # Dash 함수를 가져와 Web을 구성
# app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# ###########################################################################
# # 간단한 Layout 구성
# app.layout = dbc.Container(
#     [
#         html.H3(children='Loan Data Dash'),
#         dbc.Col(children= [html.Div(dcc.Graph(figure=bubble_fig))] )
#     ]
# )
# ###########################################################################
# # 앞서 구성된 Layout을 Web으로 구동

# output.enable_custom_widget_manager()

# app.run(mode='inline')


In [10]:
import plotly.io as pio

table_html = portfolio_df.to_html(index=False, classes="styled-table", border=0)
sorted_result = df.sort_values('종합점수', ascending=False)[['종목명', '종합점수_100', '작년 영업이익', '작년 당기순이익', '내후년 영업이익(E)', '내후년 당기순이익(E)',
       '매출액_3Y성장률', '영업이익_3Y성장률', '순이익_3Y성장률', '시가총액', '영업이익_PER', '순이익_PER','투자스타일']]
result_html = sorted_result.to_html(index=False, classes="styled-table", border=0)
# 각 그래프를 HTML fragment로 변환
bar_html = pio.to_html(fig, include_plotlyjs=False, full_html=False)
bubble_html = pio.to_html(bubble_fig, include_plotlyjs=False, full_html=False)
heatmap_html = pio.to_html(heatmap_fig, include_plotlyjs=False, full_html=False)

# 전체 HTML 템플릿
html_template = f"""
<html>
<head>
    <meta charset="utf-8">
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <title>Quant Multi-Factor Dashboard</title>
    <style>
        body {{
            font-family: Arial;
            background-color: #f5f7fa;
            margin: 40px;
        }}
        h1 {{
            text-align: center;
        }}
        .chart-container {{
            margin-bottom: 80px;
            padding: 20px;
            background: white;
            border-radius: 12px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.1);
        }}
        .styled-table {{
            border-collapse: collapse;
            margin: 25px 0;
            font-size: 15px;
            width: 100%;
        }}
        .styled-table thead tr {{
            background-color: #08306b;
            color: white;
            text-align: center;
        }}
        .styled-table th, .styled-table td {{
            padding: 12px 15px;
            text-align: center;
        }}
        .styled-table tbody tr:nth-child(even) {{
            background-color: #f3f3f3;
        }}
    </style>
</head>
<body>

<h1>📊 승호표 Dashboard</h1>

<div class="chart-container">
<h2> 📊성장율 / PER 랭킹</h2>
{bar_html}
</div>

<div class="chart-container">
<h2>📊 멀티팩터 점수 히트맵</h2>
{heatmap_html}
</div>

<div class="chart-container">
<h2>📈 성장 vs PER 버블차트</h2>
{bubble_html}
</div>

<div class="chart-container">
<h2>🏆 종합점수 TOP10 1000만원 투자예시</h2>
{table_html}
</div>

<div class="chart-container">
<h2>(Appendix) 계산 결과</h2>
{result_html}
</div>

</body>
</html>
"""

# 파일 저장
file_path = "/content/quant_dashboard.html"

with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_template)

print("✅ 생성 완료:", file_path)


✅ 생성 완료: /content/quant_dashboard.html
